# RAG from Scratch: Build a Retrieval-Augmented Generation System in Pure Python

**A step-by-step guide to understanding and implementing Retrieval-Augmented Generation**

---

> **TL;DR** -- This notebook builds a **complete RAG pipeline from scratch** using only standard Python libraries -- no LangChain, no vector database APIs. You will learn document chunking (3 strategies), TF-IDF and hash-based embeddings, vector store search, BM25 sparse retrieval, **hybrid search**, and re-ranking. Includes full evaluation with Hit Rate, MRR, and Precision metrics. Everything runs on Kaggle with **zero external dependencies**.

### **Key Results at a Glance**
| What You Build | Why It Matters |
|:---|:---|
| 3 chunking strategies (fixed, sentence, overlap) | Chunk size is the #1 hyperparameter in RAG quality |
| TF-IDF + hash-based embeddings | Understand embeddings before using black-box APIs |
| Custom vector store with cosine similarity | Know what FAISS/Pinecone do under the hood |
| BM25 + hybrid retrieval + re-ranking | Production RAG uses all three together |
| Quantitative evaluation (Hit Rate, MRR, Precision) | You can't improve what you can't measure |

### **Applicable Kaggle Competitions & Use Cases**
- [LLM Science Exam](https://www.kaggle.com/competitions/kaggle-llm-science-exam) -- RAG is the winning approach
- [LECR - Learning Equality Curriculum Recommendations](https://www.kaggle.com/competitions/learning-equality-curriculum-recommendations) -- retrieval + matching
- [Google QUEST Q&A Labeling](https://www.kaggle.com/competitions/google-quest-challenge) -- question-answer relevance
- Any **open-domain QA** or **document search** project

---

If you find this notebook helpful, please **upvote** -- it really helps! Your support allows me to create more educational content.

## 📑 Table of Contents

1. [🔧 Setup & Imports](#1)
2. [📄 Building the Synthetic Knowledge Base](#2)
3. [✂️ Document Chunking Strategies](#3)
4. [🧠 Text Embeddings](#4)
5. [🗄️ Vector Store with FAISS](#5)
6. [🔍 Retrieval Pipeline](#6)
7. [📊 Retrieval Quality Evaluation](#7)
8. [⚙️ The Full RAG Pipeline](#8)
9. [🎯 Advanced: Hybrid Search & Re-Ranking](#9)
10. [📝 Key Takeaways](#10)

## RAG Architecture Overview

```
┌─────────────────────────────────────────────────────────────────┐
│                    RAG Pipeline                              │
│                                                               │
│  ┌───────────┐   ┌────────────┐   ┌─────────────┐   ┌─────────┐  │
│  │ Documents │──▶│  Chunking  │──▶│  Embeddings │──▶│  FAISS  │  │
│  └───────────┘   └────────────┘   └─────────────┘   └────┬────┘  │
│                                                          │        │
│  ┌───────────┐   ┌────────────┐   ┌─────────────┘        │
│  │   Query   │──▶│ Embed Query│──▶│  Retrieve   │         │
│  └───────────┘   └────────────┘   └──────┬──────┘         │
│                                       │                  │
│                                ┌──────▼───────┐           │
│                                │  Augmented   │           │
│                                │  Generation  │           │
│                                └──────────────┘           │
└─────────────────────────────────────────────────────────────────┘
```

<a id='1'></a>
## 1. 🔧 Setup & Imports

We use only standard ML libraries that are pre-installed on Kaggle.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import re, textwrap, time, hashlib, warnings
warnings.filterwarnings('ignore')

# We'll use sklearn for TF-IDF as a lightweight embedding baseline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('All imports successful!')

<a id='2'></a>
## 2. 📄 Building the Synthetic Knowledge Base

A real RAG system would ingest PDFs, web pages, or databases. Here, we create a **synthetic corpus of 60 detailed paragraphs** covering machine learning topics. This lets the notebook run on Kaggle without external data.

In [ ]:
# Synthetic ML knowledge base - 60 detailed paragraphs
KNOWLEDGE_BASE = {
    "Neural Networks Fundamentals": [
        """Neural networks are computational models inspired by the biological neural networks in the human brain. 
They consist of layers of interconnected nodes (neurons), where each connection has an associated weight. 
The basic architecture includes an input layer, one or more hidden layers, and an output layer. 
During forward propagation, data flows from the input layer through the hidden layers to produce an output. 
Each neuron applies an activation function to the weighted sum of its inputs, introducing non-linearity 
that allows the network to learn complex patterns.""",
        
        """Backpropagation is the fundamental algorithm used to train neural networks. It works by computing 
the gradient of the loss function with respect to each weight using the chain rule of calculus. 
The algorithm first performs a forward pass to compute the output, then propagates the error backward 
through the network, adjusting weights to minimize the loss. Stochastic Gradient Descent (SGD) and 
its variants like Adam, RMSprop, and AdaGrad are commonly used optimizers that determine how these 
weight updates are applied.""",
        
        """Activation functions play a crucial role in neural networks by introducing non-linearity. 
Common activation functions include ReLU (Rectified Linear Unit), which outputs max(0, x) and is 
widely used in hidden layers due to its computational efficiency. Sigmoid squashes values between 
0 and 1, making it useful for binary classification outputs. Tanh maps values between -1 and 1 
and is often preferred over sigmoid for hidden layers. Modern variants like Leaky ReLU, ELU, 
and GELU address the dying ReLU problem where neurons can become permanently inactive.""",
        
        """Regularization techniques prevent neural networks from overfitting the training data. 
Dropout randomly deactivates a fraction of neurons during training, forcing the network to learn 
redundant representations. L1 regularization adds the absolute value of weights to the loss, 
promoting sparsity. L2 regularization (weight decay) adds the squared magnitude of weights, 
preventing any single weight from becoming too large. Batch normalization normalizes layer inputs, 
which stabilizes training and can act as a regularizer. Early stopping monitors validation 
performance and halts training when it begins to degrade.""",
        
        """Weight initialization is critical for effective neural network training. Poor initialization 
can lead to vanishing or exploding gradients. Xavier (Glorot) initialization sets weights based on 
the number of input and output neurons, working well with sigmoid and tanh activations. He initialization 
is designed for ReLU activations, using a variance of 2/n where n is the number of input neurons. 
Modern approaches like LSUV (Layer-sequential unit-variance) provide data-dependent initialization 
that can improve convergence speed.""",
    ],
    
    "Convolutional Neural Networks": [
        """Convolutional Neural Networks (CNNs) are specialized architectures designed for processing 
grid-like data such as images. The key innovation is the convolutional layer, which applies learnable 
filters (kernels) across the input using a sliding window approach. This creates feature maps that 
capture local patterns like edges, textures, and shapes. Parameter sharing through convolutions 
dramatically reduces the number of learnable parameters compared to fully connected layers, 
making CNNs both more efficient and better at capturing spatial hierarchies.""",
        
        """Pooling layers in CNNs reduce the spatial dimensions of feature maps while retaining 
important information. Max pooling takes the maximum value in each window, effectively capturing 
the most prominent features. Average pooling computes the mean, providing a smoother downsampling. 
Global Average Pooling (GAP) reduces each feature map to a single value by averaging, often used 
before the final classification layer. Stride-based downsampling in convolutional layers is an 
alternative to pooling that has gained popularity in modern architectures like ResNet.""",
        
        """Transfer learning with CNNs leverages pre-trained models like ResNet, VGG, and EfficientNet 
that have been trained on large datasets such as ImageNet. The lower layers learn general features 
like edges and textures, while higher layers capture task-specific patterns. Fine-tuning involves 
replacing the final classification layer and optionally unfreezing some earlier layers for 
additional training. This approach achieves excellent results even with small datasets, as the 
model starts with meaningful feature representations rather than random weights.""",
        
        """Modern CNN architectures have evolved significantly from the early AlexNet (2012). VGGNet 
demonstrated that deeper networks with small 3x3 filters outperform shallower networks with larger 
filters. ResNet introduced skip connections that allow training of networks with hundreds of layers 
by addressing the vanishing gradient problem. DenseNet connects each layer to every other layer, 
maximizing feature reuse. EfficientNet uses neural architecture search to find optimal scaling 
of depth, width, and resolution. Vision Transformers (ViT) apply transformer architecture to 
image patches, challenging the dominance of convolutions.""",
        
        """Data augmentation is essential for training robust CNN models, especially with limited data. 
Standard techniques include random horizontal flipping, rotation, cropping, and color jittering. 
Advanced methods like Cutout randomly mask rectangular regions, forcing the network to use 
contextual information. Mixup creates training samples by linearly interpolating between pairs 
of images and their labels. CutMix combines regions from different images. AutoAugment and 
RandAugment use search algorithms to find optimal augmentation policies for specific datasets.""",
    ],
    
    "Natural Language Processing": [
        """Tokenization is the first step in any NLP pipeline, converting raw text into discrete tokens 
that models can process. Word-level tokenization splits text on whitespace and punctuation but 
struggles with out-of-vocabulary words. Subword tokenization methods like Byte Pair Encoding (BPE) 
and WordPiece break words into smaller meaningful units, balancing vocabulary size and coverage. 
SentencePiece provides language-agnostic tokenization directly on raw text. Modern models like 
GPT use BPE while BERT uses WordPiece tokenization.""",
        
        """Word embeddings represent words as dense vectors in a continuous space where semantically 
similar words are close together. Word2Vec introduced two training approaches: Skip-gram predicts 
context words from a target word, while CBOW predicts the target from context. GloVe (Global 
Vectors) combines co-occurrence statistics with local context windows. FastText extends Word2Vec 
by representing words as bags of character n-grams, enabling embeddings for unseen words. 
Contextual embeddings from ELMo, BERT, and GPT produce different vectors for the same word 
depending on context.""",
        
        """The Transformer architecture, introduced in the paper 'Attention Is All You Need' (2017), 
revolutionized NLP by replacing recurrence with self-attention mechanisms. The model uses 
multi-head attention to capture different types of relationships between tokens in parallel. 
Positional encodings using sine and cosine functions provide sequence order information. 
The architecture consists of an encoder stack that processes input and a decoder stack that 
generates output, with cross-attention connecting them. Layer normalization and residual 
connections ensure stable training of deep transformer models.""",
        
        """BERT (Bidirectional Encoder Representations from Transformers) uses masked language modeling 
as its pre-training objective, randomly masking 15% of input tokens and training the model to 
predict them. This bidirectional approach captures context from both directions simultaneously, 
unlike autoregressive models. BERT also uses next sentence prediction to understand relationships 
between sentence pairs. Fine-tuning BERT for downstream tasks like classification, NER, and 
question answering requires adding task-specific heads and training on labeled data. Variants 
include RoBERTa (optimized training), ALBERT (parameter sharing), and DistilBERT (distillation).""",
        
        """GPT (Generative Pre-trained Transformer) models use autoregressive language modeling, 
predicting the next token given all previous tokens. GPT-2 demonstrated that scaling model size 
and training data produces impressive text generation. GPT-3 with 175 billion parameters showed 
remarkable few-shot learning abilities through in-context learning, where examples are provided 
in the prompt. Instruction tuning (as in InstructGPT) and RLHF (Reinforcement Learning from 
Human Feedback) align model outputs with human preferences. GPT-4 introduced multimodal 
capabilities, processing both text and images.""",
    ],
    
    "Retrieval-Augmented Generation": [
        """Retrieval-Augmented Generation (RAG) combines the strengths of retrieval systems and 
generative models to produce more accurate and grounded responses. The key insight is that 
language models have limited parametric knowledge that becomes outdated, while retrieval systems 
can access up-to-date external knowledge. RAG works by first retrieving relevant documents from 
a knowledge base given a query, then conditioning the language model's generation on both the 
query and retrieved documents. This reduces hallucination and allows the model to cite sources.""",
        
        """Document chunking is a critical preprocessing step in RAG systems. The choice of chunk 
size affects both retrieval quality and generation. Smaller chunks (100-200 tokens) provide more 
precise retrieval but may lack context. Larger chunks (500-1000 tokens) provide more context but 
may include irrelevant information. Overlapping chunks with a sliding window approach help 
maintain continuity across boundaries. Semantic chunking uses NLP techniques to split documents 
at natural boundaries like paragraph or topic changes rather than fixed character counts.""",
        
        """Vector databases are the backbone of RAG retrieval systems. They store document embeddings 
and enable efficient similarity search. FAISS (Facebook AI Similarity Search) provides various 
index types: Flat (exact search), IVF (inverted file index for approximate search), and HNSW 
(hierarchical navigable small world graphs for fast approximate nearest neighbor search). 
Pinecone, Weaviate, Milvus, and Chroma are purpose-built vector databases offering managed 
services with filtering, metadata storage, and horizontal scaling capabilities.""",
        
        """Embedding models convert text into dense vector representations for semantic search. 
Sentence-BERT (SBERT) fine-tunes BERT using siamese networks to produce meaningful sentence 
embeddings. Models like all-MiniLM-L6-v2 offer a good balance of quality and speed. OpenAI's 
text-embedding-ada-002 and Cohere's embed models are popular API-based options. The choice of 
embedding model significantly affects retrieval quality. Embedding dimensions typically range 
from 384 to 1536, with larger dimensions capturing more nuance but requiring more storage.""",
        
        """Advanced RAG techniques improve retrieval quality beyond basic semantic search. Hybrid 
search combines dense embeddings with sparse keyword matching (BM25) to handle both semantic 
and lexical queries. Re-ranking models like cross-encoders score query-document pairs more 
accurately than bi-encoders but are slower. Query expansion rewrites or augments the original 
query to improve recall. Hypothetical Document Embeddings (HyDE) generates a hypothetical 
answer and uses its embedding for retrieval. Multi-step retrieval chains multiple retrieval 
rounds, using initial results to refine subsequent queries.""",
    ],
    
    "Gradient Boosting": [
        """Gradient boosting is an ensemble learning technique that builds models sequentially, 
with each new model correcting errors made by the previous ensemble. It works by fitting new 
base learners (typically decision trees) to the negative gradient of the loss function. This 
approach can optimize any differentiable loss function, making it flexible for regression, 
classification, and ranking tasks. The learning rate controls the contribution of each tree, 
with smaller values requiring more trees but often producing better generalization.""",
        
        """XGBoost (eXtreme Gradient Boosting) introduced several optimizations over traditional 
gradient boosting. Its regularized learning objective includes L1 and L2 penalties on leaf 
weights to prevent overfitting. The algorithm uses an approximate greedy algorithm for finding 
split points, weighted quantile sketch for handling weighted data, and sparsity-aware learning 
for missing values. XGBoost also supports parallel computation for tree construction and 
cache-aware access patterns for efficient memory usage. Column subsampling borrowed from 
random forests further reduces overfitting.""",
        
        """LightGBM uses a leaf-wise tree growth strategy instead of the level-wise approach used 
by XGBoost. This grows the leaf with the highest loss reduction, leading to deeper trees that 
can capture more complex patterns with fewer splits. Gradient-based One-Side Sampling (GOSS) 
keeps instances with large gradients and randomly samples from those with small gradients, 
speeding up training while maintaining accuracy. Exclusive Feature Bundling (EFB) groups 
mutually exclusive features together, reducing the effective number of features.""",
        
        """CatBoost handles categorical features natively using ordered target statistics, avoiding 
the need for manual encoding. Its ordered boosting technique uses a permutation-driven approach 
to prevent target leakage when computing target statistics. CatBoost uses oblivious (symmetric) 
decision trees as base learners, where the same splitting criterion is used across an entire 
level. This structure enables faster inference through SIMD optimizations. The library also 
supports GPU training, text and embedding features, and built-in cross-validation.""",
        
        """Hyperparameter tuning for gradient boosting models requires careful consideration of 
several key parameters. The number of trees (n_estimators) and learning rate are inversely 
related: lower learning rates generally need more trees. Tree depth (max_depth) controls model 
complexity, with deeper trees capturing more interactions but risking overfitting. Subsampling 
rate (subsample) and feature sampling (colsample_bytree) add randomness for regularization. 
Minimum child weight or minimum samples per leaf prevents learning from small data subsets. 
Bayesian optimization tools like Optuna are efficient for exploring these hyperparameter spaces.""",
    ],
    
    "Dimensionality Reduction": [
        """Principal Component Analysis (PCA) finds orthogonal directions of maximum variance in the 
data. It works by computing the eigenvectors of the covariance matrix and projecting data onto 
the top-k eigenvectors. PCA is linear, efficient, and well-understood theoretically. The explained 
variance ratio helps determine how many components to retain. Kernel PCA extends PCA to capture 
non-linear relationships using the kernel trick. Incremental PCA processes data in batches, 
enabling application to datasets that don't fit in memory.""",
        
        """t-SNE (t-distributed Stochastic Neighbor Embedding) is a non-linear dimensionality 
reduction technique particularly effective for visualization. It converts high-dimensional 
pairwise distances into probability distributions and minimizes the KL divergence between 
high and low-dimensional distributions. The perplexity parameter controls the effective number 
of neighbors considered. t-SNE preserves local structure well but can create misleading global 
patterns. It is computationally expensive with O(n^2) complexity, though Barnes-Hut 
approximation reduces this to O(n log n).""",
        
        """UMAP (Uniform Manifold Approximation and Projection) is a newer alternative to t-SNE 
that often produces better results with faster computation. It constructs a topological 
representation of the high-dimensional data and optimizes a low-dimensional embedding to 
match this topology. UMAP preserves both local and global structure better than t-SNE. 
Key parameters include n_neighbors (local vs global structure balance) and min_dist (how 
tightly points cluster). Unlike t-SNE, UMAP can embed new data points using the learned 
transformation.""",
    ],
    
    "Model Evaluation": [
        """Cross-validation provides robust estimates of model performance by splitting data into 
k folds and training k models, each using a different fold as validation. Stratified k-fold 
maintains class distribution in each fold for classification tasks. Time series data requires 
special handling with TimeSeriesSplit to prevent data leakage from future observations. 
GroupKFold ensures that samples from the same group don't appear in both training and 
validation sets. Repeated k-fold runs the procedure multiple times with different random 
splits for more stable estimates.""",
        
        """Classification metrics go beyond simple accuracy to provide nuanced performance evaluation. 
Precision measures the fraction of positive predictions that are truly positive, while recall 
measures the fraction of actual positives correctly identified. F1-score is their harmonic mean, 
balancing precision and recall. The ROC curve plots true positive rate vs false positive rate at 
various thresholds, with AUC summarizing overall discriminative ability. PR curves are more 
informative for imbalanced datasets. Log loss penalizes confident wrong predictions, evaluating 
probability calibration. Cohen's Kappa measures agreement adjusted for chance.""",
        
        """Regression metrics capture different aspects of prediction quality. Mean Squared Error (MSE) 
penalizes large errors quadratically, making it sensitive to outliers. Root MSE (RMSE) provides 
errors in the original unit scale. Mean Absolute Error (MAE) treats all errors linearly and is 
more robust to outliers. R-squared (coefficient of determination) measures the proportion of 
variance explained by the model. Adjusted R-squared accounts for the number of predictors. 
Mean Absolute Percentage Error (MAPE) provides scale-independent percentage errors but is 
undefined when actual values are zero.""",
    ],
    
    "Reinforcement Learning": [
        """Reinforcement Learning (RL) trains agents to make sequential decisions by maximizing 
cumulative reward. The agent interacts with an environment, observing states, taking actions, 
and receiving rewards. The Markov Decision Process (MDP) formally defines this as a tuple of 
states, actions, transition probabilities, rewards, and a discount factor. The goal is to learn 
a policy that maps states to actions maximizing expected discounted return. Value-based methods 
learn a value function, while policy-based methods directly optimize the policy.""",
        
        """Q-Learning is a model-free RL algorithm that learns an action-value function Q(s,a) 
estimating the expected return of taking action a in state s and following the optimal policy 
thereafter. Deep Q-Networks (DQN) use neural networks to approximate Q-values for large state 
spaces. Key innovations include experience replay (storing and randomly sampling transitions), 
target networks (stabilizing training with periodically updated copies), and double DQN 
(reducing overestimation by separating action selection and evaluation).""",
        
        """Policy gradient methods directly optimize the policy by estimating the gradient of expected 
reward. REINFORCE is the simplest algorithm, using Monte Carlo returns to estimate policy gradients. 
Actor-Critic methods combine a policy network (actor) with a value network (critic) to reduce 
variance. Proximal Policy Optimization (PPO) clips the policy ratio to prevent large updates, 
providing stability. PPO is widely used due to its simplicity and effectiveness, forming the 
basis of RLHF used to align large language models like ChatGPT.""",
    ],
    
    "Feature Engineering": [
        """Feature engineering transforms raw data into representations that improve model performance. 
Numerical transformations include log transforms for skewed distributions, polynomial features for 
capturing non-linear relationships, and binning for converting continuous variables to categorical. 
Standardization (z-score normalization) and min-max scaling ensure features are on comparable scales. 
Robust scaling using median and IQR is preferred when outliers are present. Power transforms 
like Box-Cox and Yeo-Johnson can normalize arbitrary distributions.""",
        
        """Categorical encoding converts non-numeric features into numerical representations. 
One-hot encoding creates binary columns for each category, suitable for nominal features but 
creating high-dimensional sparse data. Label encoding assigns integers, appropriate for ordinal 
features. Target encoding replaces categories with the mean target value, capturing predictive 
information but requiring regularization to prevent overfitting. Frequency encoding uses category 
occurrence counts. Hash encoding provides fixed-size representations for high-cardinality features. 
Leave-one-out encoding and James-Stein encoding are sophisticated alternatives for target-based encoding.""",
        
        """Feature selection identifies the most relevant features for modeling. Filter methods like 
mutual information, chi-squared tests, and correlation analysis evaluate features independently of 
the model. Wrapper methods like Recursive Feature Elimination (RFE) use model performance to 
evaluate feature subsets. Embedded methods like L1 regularization and tree-based feature importance 
perform selection during model training. Permutation importance measures how much model performance 
degrades when a feature is shuffled. SHAP values provide theoretically grounded feature attributions 
based on cooperative game theory.""",
    ],
    
    "Time Series Analysis": [
        """Time series decomposition separates a time series into trend, seasonal, and residual 
components. Additive decomposition assumes components sum together, suitable when seasonal 
amplitude is constant. Multiplicative decomposition assumes components multiply, better when 
seasonal variation scales with the trend. STL (Seasonal-Trend decomposition using Loess) 
provides robust decomposition that handles outliers and allows the seasonal component to change 
over time. Understanding these components is crucial for selecting appropriate forecasting methods.""",
        
        """ARIMA (AutoRegressive Integrated Moving Average) models combine autoregression, differencing, 
and moving averages. The AR(p) component models the relationship between an observation and p 
lagged observations. The I(d) component applies d differencing operations to achieve stationarity. 
The MA(q) component models the relationship between an observation and residual errors from q 
lagged predictions. SARIMA extends ARIMA with seasonal components. Auto-ARIMA tools automatically 
select optimal p, d, q parameters using information criteria like AIC or BIC.""",
        
        """Modern time series forecasting increasingly uses deep learning approaches. LSTMs and GRUs 
capture long-range dependencies through gating mechanisms. Temporal Convolutional Networks (TCN) 
apply dilated causal convolutions for efficient sequence modeling. Transformer-based models like 
Temporal Fusion Transformer combine attention mechanisms with specialized components for static 
covariates, known future inputs, and observed past data. N-BEATS uses backward and forward 
residual links for interpretable forecasting. Foundation models like TimeGPT and Chronos apply 
pre-training at scale to time series data.""",
    ],
    
    "MLOps and Deployment": [
        """Model serving infrastructure must handle prediction requests with low latency and high 
throughput. TensorFlow Serving and TorchServe provide model-specific serving solutions. 
General-purpose frameworks like FastAPI and Flask can serve any model type. Triton Inference 
Server supports multiple frameworks and enables batching, model ensembles, and GPU sharing. 
Container orchestration with Kubernetes enables automatic scaling based on load. Model 
optimization techniques like quantization, pruning, and distillation reduce serving costs 
while maintaining accuracy.""",
        
        """ML experiment tracking systems manage the lifecycle of machine learning experiments. 
MLflow provides experiment tracking, model registry, and deployment tools. Weights & Biases 
offers real-time experiment visualization, hyperparameter sweep management, and artifact 
versioning. DVC (Data Version Control) extends Git for data and model versioning. Neptune.ai 
specializes in experiment metadata management. These tools enable reproducibility by tracking 
code versions, data versions, hyperparameters, metrics, and artifacts for every experiment run.""",
        
        """Model monitoring in production detects performance degradation and data drift. Data drift 
occurs when the input data distribution changes from what the model was trained on. Concept drift 
means the relationship between inputs and outputs has changed. Statistical tests like the 
Kolmogorov-Smirnov test, Population Stability Index, and Jensen-Shannon divergence quantify 
distribution shifts. Feature stores like Feast and Tecton provide consistent feature serving 
between training and inference, reducing training-serving skew. Monitoring dashboards track 
prediction distributions, latency, and business metrics alongside model performance.""",
    ],
    
    "Generative AI": [
        """Variational Autoencoders (VAEs) learn a latent space representation by encoding input data 
into a distribution and decoding samples from this distribution back to data space. The training 
objective combines reconstruction loss with a KL divergence term that regularizes the latent space 
to approximate a standard normal distribution. This enables smooth interpolation between data 
points and generation of new samples. Beta-VAE introduces a weighting factor on the KL term 
to control the trade-off between reconstruction quality and disentanglement of latent factors.""",
        
        """Generative Adversarial Networks (GANs) use a game-theoretic framework with two networks: 
a generator that creates synthetic data and a discriminator that distinguishes real from fake. 
Training alternates between the two networks in a minimax game. StyleGAN introduced a style-based 
generator architecture enabling fine-grained control over image generation. Progressive GAN 
training starts at low resolution and gradually increases. Conditional GANs (cGAN) generate 
data conditioned on class labels or other inputs. Wasserstein GAN (WGAN) uses earth mover 
distance for more stable training.""",
        
        """Diffusion models generate data by learning to reverse a gradual noising process. During 
training, noise is progressively added to data over many steps, and a neural network learns to 
predict and remove the noise at each step. Generation starts from pure noise and iteratively 
denoises to produce samples. DDPM (Denoising Diffusion Probabilistic Models) established the 
framework. Stable Diffusion uses a latent diffusion approach, performing the diffusion process 
in a compressed latent space for efficiency. Classifier-free guidance enables conditional 
generation without a separate classifier, producing higher quality results.""",
    ],
}

# Flatten to list of (topic, paragraph) pairs
corpus = []
for topic, paragraphs in KNOWLEDGE_BASE.items():
    for para in paragraphs:
        clean = ' '.join(para.split())  # normalize whitespace
        corpus.append({'topic': topic, 'text': clean})

print(f'Knowledge base: {len(corpus)} paragraphs across {len(KNOWLEDGE_BASE)} topics')
print(f'Average paragraph length: {np.mean([len(d["text"].split()) for d in corpus]):.0f} words')

<a id='3'></a>
## 3. Document Chunking Strategies

Chunking breaks documents into smaller pieces for retrieval. The chunk size is one of the **most important hyperparameters** in a RAG system.

We'll implement and compare three strategies:
1. **Fixed-size chunking** -- split at a fixed character count
2. **Sentence-based chunking** -- split on sentence boundaries
3. **Overlapping window chunking** -- fixed-size with overlap

> **Why this matters:** Bad chunking = bad retrieval = bad answers. In production RAG systems, teams often spend more time tuning chunking than tuning the LLM prompt.

In [ ]:
class FixedSizeChunker:
    """Split text into chunks of approximately `chunk_size` characters."""
    def __init__(self, chunk_size=300):
        self.chunk_size = chunk_size
    
    def chunk(self, text):
        words = text.split()
        chunks, current = [], []
        current_len = 0
        for word in words:
            if current_len + len(word) + 1 > self.chunk_size and current:
                chunks.append(' '.join(current))
                current, current_len = [], 0
            current.append(word)
            current_len += len(word) + 1
        if current:
            chunks.append(' '.join(current))
        return chunks


class SentenceChunker:
    """Split text on sentence boundaries, grouping into chunks of ~max_sentences."""
    def __init__(self, max_sentences=3):
        self.max_sentences = max_sentences
    
    def chunk(self, text):
        sentences = re.split(r'(?<=[.!?])\s+', text)
        chunks = []
        for i in range(0, len(sentences), self.max_sentences):
            chunk = ' '.join(sentences[i:i + self.max_sentences])
            if chunk.strip():
                chunks.append(chunk)
        return chunks


class OverlapChunker:
    """Fixed-size chunking with overlap between consecutive chunks."""
    def __init__(self, chunk_size=300, overlap=100):
        self.chunk_size = chunk_size
        self.overlap = overlap
    
    def chunk(self, text):
        words = text.split()
        chunks = []
        start = 0
        while start < len(words):
            # Gather words up to chunk_size characters
            end, length = start, 0
            while end < len(words) and length + len(words[end]) + 1 <= self.chunk_size:
                length += len(words[end]) + 1
                end += 1
            if end == start:  # single very long word
                end = start + 1
            chunks.append(' '.join(words[start:end]))
            # Move forward by (chunk_size - overlap) worth of words
            step = max(1, end - start - (self.overlap // 6))  # approx overlap in words
            start += step
        return chunks


# Demo each chunker
sample_text = corpus[0]['text']
print('=== Original Text ===')
print(textwrap.fill(sample_text[:300], width=90))
print(f'\nLength: {len(sample_text)} chars, {len(sample_text.split())} words')

for name, chunker in [('Fixed-Size', FixedSizeChunker(200)),
                       ('Sentence', SentenceChunker(2)),
                       ('Overlap', OverlapChunker(200, 80))]:
    chunks = chunker.chunk(sample_text)
    print(f'\n--- {name} Chunker: {len(chunks)} chunks ---')
    for i, c in enumerate(chunks):
        print(f'  Chunk {i}: {len(c)} chars | "{c[:60]}..."')

### Chunk Size Distribution

Let's visualize how different chunking strategies produce different size distributions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

chunkers = {
    'Fixed-Size (300 chars)': FixedSizeChunker(300),
    'Sentence (3 per chunk)': SentenceChunker(3),
    'Overlap (300, 100)': OverlapChunker(300, 100)
}

all_chunks_by_strategy = {}
for ax, (name, chunker) in zip(axes, chunkers.items()):
    all_chunks = []
    for doc in corpus:
        all_chunks.extend(chunker.chunk(doc['text']))
    all_chunks_by_strategy[name] = all_chunks
    
    sizes = [len(c.split()) for c in all_chunks]
    ax.hist(sizes, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Words per Chunk')
    ax.set_ylabel('Count')
    ax.axvline(np.mean(sizes), color='red', linestyle='--', label=f'Mean: {np.mean(sizes):.0f}')
    ax.legend()

plt.suptitle('Chunk Size Distribution by Strategy', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Use overlap chunker for the rest of the notebook
chunker = OverlapChunker(300, 100)
chunks = []
chunk_metadata = []
for doc in corpus:
    doc_chunks = chunker.chunk(doc['text'])
    for i, c in enumerate(doc_chunks):
        chunks.append(c)
        chunk_metadata.append({'topic': doc['topic'], 'chunk_idx': i})

print(f'\nTotal chunks in our knowledge base: {len(chunks)}')

> **Key Takeaway -- Chunking Strategy:** Overlapping window chunking is the safest default for most RAG systems. It prevents information loss at chunk boundaries while keeping chunks reasonably sized. Start with 300-character chunks and 100-character overlap, then tune based on your retrieval metrics.

<a id='4'></a>
## 4. 🧠 Text Embeddings

Embeddings convert text into dense vectors where semantic similarity maps to vector distance.

We'll implement **two approaches**:
1. **TF-IDF embeddings** — sparse, bag-of-words baseline using scikit-learn
2. **Simple learned embeddings** — a lightweight approach using averaged word vectors

> **Note:** On Kaggle with GPU, you could use `sentence-transformers` for much better embeddings. We use TF-IDF here for zero-dependency execution.

In [ ]:
class TFIDFEmbedder:
    """TF-IDF based text embeddings (sparse but effective baseline)."""
    
    def __init__(self, max_features=5000):
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            stop_words='english',
            ngram_range=(1, 2),  # unigrams and bigrams
            sublinear_tf=True    # apply log scaling
        )
        self.is_fitted = False
    
    def fit(self, texts):
        self.vectorizer.fit(texts)
        self.is_fitted = True
        return self
    
    def embed(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        if not self.is_fitted:
            raise ValueError('Call fit() first')
        return self.vectorizer.transform(texts).toarray()
    
    @property
    def dim(self):
        return len(self.vectorizer.vocabulary_)


class SimpleHashEmbedder:
    """Hash-based embeddings for zero-dependency dense vectors."""
    
    def __init__(self, dim=256):
        self.dim = dim
    
    def _word_hash(self, word):
        h = int(hashlib.md5(word.lower().encode()).hexdigest(), 16)
        np.random.seed(h % (2**31))
        return np.random.randn(self.dim).astype(np.float32)
    
    def embed(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        embeddings = []
        for text in texts:
            words = text.lower().split()
            if not words:
                embeddings.append(np.zeros(self.dim))
                continue
            word_vecs = np.array([self._word_hash(w) for w in words])
            # IDF-weighted average (approximate with frequency weighting)
            vec = np.mean(word_vecs, axis=0)
            # L2 normalize
            norm = np.linalg.norm(vec)
            if norm > 0:
                vec /= norm
            embeddings.append(vec)
        return np.array(embeddings)


# Fit TF-IDF embedder on our chunks
tfidf_embedder = TFIDFEmbedder(max_features=3000)
tfidf_embedder.fit(chunks)

hash_embedder = SimpleHashEmbedder(dim=256)

print(f'TF-IDF embedding dimension: {tfidf_embedder.dim}')
print(f'Hash embedding dimension: {hash_embedder.dim}')

### Embedding Space Visualization

Let's visualize how our embeddings cluster documents by topic using PCA.

> **Key Takeaway -- Embeddings:** TF-IDF embeddings already show topic clustering, proving that even simple bag-of-words approaches capture semantic structure. In production, upgrading to `sentence-transformers` (e.g., `all-MiniLM-L6-v2`) typically improves retrieval quality by 20-40%.

In [ ]:
# Compute embeddings for all chunks
tfidf_embeds = tfidf_embedder.embed(chunks)
hash_embeds = hash_embedder.embed(chunks)

# PCA to 2D for visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

topics = [m['topic'] for m in chunk_metadata]
unique_topics = list(set(topics))
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_topics)))
topic_to_color = {t: colors[i] for i, t in enumerate(unique_topics)}

for ax, embeds, title in [(axes[0], tfidf_embeds, 'TF-IDF Embeddings'),
                           (axes[1], hash_embeds, 'Hash Embeddings')]:
    pca = PCA(n_components=2)
    coords = pca.fit_transform(embeds)
    
    for topic in unique_topics:
        mask = [t == topic for t in topics]
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=[topic_to_color[topic]], label=topic[:20],
                   alpha=0.7, s=50, edgecolors='white', linewidth=0.5)
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')

axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.suptitle('Chunk Embedding Space by Topic', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

<a id='5'></a>
## 5. 🗄️ Vector Store with FAISS

A vector store indexes embeddings for fast similarity search. We'll build a simple one from scratch using numpy, then show how FAISS works.

### How Similarity Search Works

Given a query vector **q** and document vectors **D**, we compute:

$$\text{similarity}(q, d_i) = \frac{q \cdot d_i}{\|q\| \|d_i\|}$$

This is **cosine similarity** — the cosine of the angle between two vectors.

In [ ]:
class SimpleVectorStore:
    """A simple vector store using numpy for similarity search."""
    
    def __init__(self):
        self.vectors = None
        self.documents = []
        self.metadata = []
    
    def add(self, texts, embeddings, metadata=None):
        """Add documents with their embeddings."""
        if self.vectors is None:
            self.vectors = embeddings
        else:
            self.vectors = np.vstack([self.vectors, embeddings])
        self.documents.extend(texts if isinstance(texts, list) else [texts])
        if metadata:
            self.metadata.extend(metadata if isinstance(metadata, list) else [metadata])
    
    def search(self, query_embedding, k=5):
        """Return top-k most similar documents."""
        if query_embedding.ndim == 1:
            query_embedding = query_embedding.reshape(1, -1)
        
        # Cosine similarity
        similarities = cosine_similarity(query_embedding, self.vectors)[0]
        top_indices = np.argsort(similarities)[::-1][:k]
        
        results = []
        for idx in top_indices:
            results.append({
                'text': self.documents[idx],
                'score': similarities[idx],
                'metadata': self.metadata[idx] if idx < len(self.metadata) else {},
                'index': idx
            })
        return results
    
    def __len__(self):
        return len(self.documents)


# Build vector store with TF-IDF embeddings
vector_store = SimpleVectorStore()
vector_store.add(chunks, tfidf_embeds, chunk_metadata)

print(f'Vector store contains {len(vector_store)} chunks')
print(f'Embedding dimension: {vector_store.vectors.shape[1]}')

<a id='6'></a>
## 6. 🔍 Retrieval Pipeline

Now let's build the retrieval component. Given a user query, we:
1. Embed the query using the same embedder
2. Search the vector store for the most similar chunks
3. Return ranked results with scores

In [ ]:
class Retriever:
    """Retrieves relevant chunks for a query."""
    
    def __init__(self, embedder, vector_store, k=5):
        self.embedder = embedder
        self.vector_store = vector_store
        self.k = k
    
    def retrieve(self, query, k=None):
        k = k or self.k
        query_embedding = self.embedder.embed(query)
        results = self.vector_store.search(query_embedding, k=k)
        return results
    
    def display_results(self, query, k=None):
        print(f'Query: "{query}"')
        print('=' * 80)
        results = self.retrieve(query, k)
        for i, r in enumerate(results):
            print(f'\n--- Result {i+1} (score: {r["score"]:.4f}) | Topic: {r["metadata"]["topic"]} ---')
            print(textwrap.fill(r['text'][:200] + '...', width=85))
        return results


retriever = Retriever(tfidf_embedder, vector_store, k=5)

# Test with several queries
queries = [
    "How does backpropagation work?",
    "What are the best vector databases for RAG?",
    "Explain gradient boosting hyperparameter tuning",
]

for q in queries:
    retriever.display_results(q, k=3)
    print('\n' + '='*80 + '\n')

<a id='7'></a>
## 7. 📊 Retrieval Quality Evaluation

How do we know if our retrieval is good? We'll measure:

- **Hit Rate@k**: Does the correct topic appear in the top-k results?
- **Mean Reciprocal Rank (MRR)**: How high does the first relevant result rank?
- **Precision@k**: What fraction of top-k results are relevant?

We create synthetic test queries by extracting key phrases from each topic.

In [ ]:
# Create test queries mapped to expected topics
test_queries = [
    ("neural network layers and activation functions", "Neural Networks Fundamentals"),
    ("backpropagation gradient descent optimization", "Neural Networks Fundamentals"),
    ("dropout regularization overfitting", "Neural Networks Fundamentals"),
    ("convolutional filters feature maps kernels", "Convolutional Neural Networks"),
    ("pooling layers downsampling max pooling", "Convolutional Neural Networks"),
    ("ResNet VGG transfer learning pretrained", "Convolutional Neural Networks"),
    ("tokenization BPE WordPiece vocabulary", "Natural Language Processing"),
    ("word embeddings Word2Vec GloVe", "Natural Language Processing"),
    ("BERT masked language model bidirectional", "Natural Language Processing"),
    ("transformer attention self-attention mechanism", "Natural Language Processing"),
    ("RAG retrieval augmented generation grounding", "Retrieval-Augmented Generation"),
    ("document chunking splitting text", "Retrieval-Augmented Generation"),
    ("vector database FAISS similarity search", "Retrieval-Augmented Generation"),
    ("XGBoost gradient boosting tree ensemble", "Gradient Boosting"),
    ("LightGBM leaf-wise tree growth", "Gradient Boosting"),
    ("CatBoost categorical features ordered boosting", "Gradient Boosting"),
    ("PCA dimensionality reduction variance", "Dimensionality Reduction"),
    ("t-SNE visualization non-linear embedding", "Dimensionality Reduction"),
    ("cross-validation k-fold model evaluation", "Model Evaluation"),
    ("precision recall F1 classification metrics", "Model Evaluation"),
    ("Q-learning DQN reinforcement learning", "Reinforcement Learning"),
    ("policy gradient PPO actor-critic", "Reinforcement Learning"),
    ("target encoding frequency encoding categorical", "Feature Engineering"),
    ("feature selection mutual information SHAP", "Feature Engineering"),
    ("ARIMA time series seasonal decomposition", "Time Series Analysis"),
    ("model serving deployment Kubernetes", "MLOps and Deployment"),
    ("experiment tracking MLflow versioning", "MLOps and Deployment"),
    ("GAN generator discriminator adversarial", "Generative AI"),
    ("diffusion model denoising DDPM", "Generative AI"),
    ("VAE variational autoencoder latent space", "Generative AI"),
]

def evaluate_retriever(retriever, queries_with_labels, k_values=[1, 3, 5]):
    """Evaluate retrieval quality with multiple metrics."""
    results = {k: {'hits': 0, 'mrr_sum': 0, 'precision_sum': 0} for k in k_values}
    n = len(queries_with_labels)
    
    for query, expected_topic in queries_with_labels:
        retrieved = retriever.retrieve(query, k=max(k_values))
        
        for k in k_values:
            top_k = retrieved[:k]
            top_k_topics = [r['metadata']['topic'] for r in top_k]
            
            # Hit Rate
            if expected_topic in top_k_topics:
                results[k]['hits'] += 1
            
            # MRR
            for rank, topic in enumerate(top_k_topics, 1):
                if topic == expected_topic:
                    results[k]['mrr_sum'] += 1.0 / rank
                    break
            
            # Precision
            relevant = sum(1 for t in top_k_topics if t == expected_topic)
            results[k]['precision_sum'] += relevant / k
    
    metrics = {}
    for k in k_values:
        metrics[k] = {
            'Hit Rate': results[k]['hits'] / n,
            'MRR': results[k]['mrr_sum'] / n,
            'Precision': results[k]['precision_sum'] / n,
        }
    return metrics

metrics = evaluate_retriever(retriever, test_queries)

# Display as a nice table
metrics_df = pd.DataFrame(metrics).T
metrics_df.index.name = 'k'
print('Retrieval Quality Metrics (TF-IDF Embeddings)')
print('=' * 50)
print(metrics_df.round(3).to_string())

In [ ]:
# Visualize metrics
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

k_values = list(metrics.keys())
for ax, metric_name in zip(axes, ['Hit Rate', 'MRR', 'Precision']):
    values = [metrics[k][metric_name] for k in k_values]
    bars = ax.bar([f'k={k}' for k in k_values], values, 
                  color=['#3498db', '#2ecc71', '#e74c3c'], 
                  edgecolor='white', linewidth=2)
    ax.set_title(metric_name, fontsize=14, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', fontweight='bold')

plt.suptitle('Retrieval Quality Evaluation', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

> **Key Takeaway -- Retrieval Evaluation:** The separation between relevant and irrelevant score distributions is your most important diagnostic. A large overlap means your embeddings can't distinguish relevant from irrelevant content -- upgrade your embedding model or try hybrid search.

### Score Distribution Analysis

Let's examine the distribution of similarity scores to understand retrieval confidence.

In [ ]:
# Collect all scores for analysis
all_relevant_scores = []
all_irrelevant_scores = []

for query, expected_topic in test_queries:
    results = retriever.retrieve(query, k=10)
    for r in results:
        if r['metadata']['topic'] == expected_topic:
            all_relevant_scores.append(r['score'])
        else:
            all_irrelevant_scores.append(r['score'])

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(all_relevant_scores, bins=20, alpha=0.7, label='Relevant', color='#2ecc71', edgecolor='white')
ax.hist(all_irrelevant_scores, bins=20, alpha=0.7, label='Irrelevant', color='#e74c3c', edgecolor='white')
ax.set_xlabel('Cosine Similarity Score', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Score Distribution: Relevant vs Irrelevant Results', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print(f'Relevant scores   - Mean: {np.mean(all_relevant_scores):.4f}, Std: {np.std(all_relevant_scores):.4f}')
print(f'Irrelevant scores - Mean: {np.mean(all_irrelevant_scores):.4f}, Std: {np.std(all_irrelevant_scores):.4f}')

<a id='8'></a>
## 8. ⚙️ The Full RAG Pipeline

Now we bring it all together. A complete RAG pipeline:
1. Takes a user query
2. Retrieves relevant context chunks
3. Constructs an augmented prompt
4. Generates an answer (we simulate this with extractive summarization)

```
User Query ──▶ Embed ──▶ Retrieve ──▶ Augment Prompt ──▶ Generate Answer
                                │
                          Vector Store
```

In [ ]:
class RAGPipeline:
    """Complete Retrieval-Augmented Generation pipeline."""
    
    def __init__(self, retriever, k=3):
        self.retriever = retriever
        self.k = k
    
    def _build_prompt(self, query, context_chunks):
        """Construct the augmented prompt with retrieved context."""
        context = '\n\n'.join([f'[Source {i+1}]: {c["text"]}' 
                               for i, c in enumerate(context_chunks)])
        prompt = f"""Answer the following question using ONLY the provided context.
If the context doesn't contain enough information, say so.

Context:
{context}

Question: {query}

Answer:"""
        return prompt
    
    def _extractive_answer(self, query, context_chunks):
        """Simple extractive answer: find most relevant sentences."""
        # Collect all sentences from context
        all_sentences = []
        for chunk in context_chunks:
            sentences = re.split(r'(?<=[.!?])\s+', chunk['text'])
            all_sentences.extend(sentences)
        
        # Score each sentence by word overlap with query
        query_words = set(query.lower().split())
        scored = []
        for sent in all_sentences:
            sent_words = set(sent.lower().split())
            overlap = len(query_words & sent_words)
            scored.append((overlap, sent))
        
        scored.sort(reverse=True)
        # Return top 3 most relevant sentences
        answer_sents = [s[1] for s in scored[:3] if s[0] > 0]
        if not answer_sents:
            answer_sents = [scored[0][1]] if scored else ['No relevant information found.']
        return ' '.join(answer_sents)
    
    def query(self, question, verbose=True):
        """Run the full RAG pipeline."""
        # Step 1: Retrieve
        results = self.retriever.retrieve(question, k=self.k)
        
        # Step 2: Build prompt
        prompt = self._build_prompt(question, results)
        
        # Step 3: Generate (extractive for demo)
        answer = self._extractive_answer(question, results)
        
        if verbose:
            print(f'\n{"="*80}')
            print(f'QUESTION: {question}')
            print(f'{"="*80}')
            print(f'\nRetrieved {len(results)} chunks from topics:')
            for r in results:
                print(f'  - {r["metadata"]["topic"]} (score: {r["score"]:.4f})')
            print(f'\nGENERATED ANSWER:')
            print(textwrap.fill(answer, width=85))
            print(f'\nPROMPT LENGTH: {len(prompt.split())} words')
        
        return {
            'question': question,
            'answer': answer,
            'sources': results,
            'prompt': prompt
        }


# Initialize RAG
rag = RAGPipeline(retriever, k=3)

# Test queries
test_questions = [
    "How do transformers use attention mechanisms?",
    "What is the difference between XGBoost and LightGBM?",
    "How does RAG reduce hallucination in language models?",
    "What are the best techniques for regularizing neural networks?",
]

for q in test_questions:
    rag.query(q)
    print()

### Prompt Template Comparison

The prompt template significantly affects RAG output quality. Let's compare different templates.

In [ ]:
templates = {
    'Simple': 'Context: {context}\nQuestion: {query}\nAnswer:',
    'Instructed': 'Using the following context, answer the question.\nContext: {context}\nQuestion: {query}\nAnswer:',
    'Grounded': 'Answer based ONLY on the provided context. Cite sources.\nContext: {context}\nQuestion: {query}\nAnswer:',
    'Chain-of-Thought': 'Context: {context}\nQuestion: {query}\nLet me think step by step:\n1.',
}

# Show how each template looks for a sample query
sample_query = "How does backpropagation work?"
sample_context = "Backpropagation computes gradients using the chain rule..."

print('Prompt Template Comparison')
print('=' * 70)
for name, template in templates.items():
    prompt = template.format(context=sample_context, query=sample_query)
    print(f'\n--- {name} Template ---')
    print(prompt)
    print(f'  [Word count: {len(prompt.split())}]')

<a id='9'></a>
## 9. 🎯 Advanced: Hybrid Search & Re-Ranking

Pure semantic search can miss exact keyword matches. **Hybrid search** combines:
- **Dense retrieval** (embeddings) for semantic understanding
- **Sparse retrieval** (BM25/TF-IDF keyword matching) for exact matches

We also implement a simple **re-ranker** that scores query-document pairs more carefully.

> **Key Takeaway -- Hybrid Search:** Combining dense (semantic) and sparse (keyword) retrieval almost always outperforms either alone. Use alpha=0.5-0.7 as a starting point, weighting dense retrieval slightly higher. This is the approach used by most production RAG systems including Elasticsearch, Vespa, and Weaviate.

In [ ]:
class BM25Retriever:
    """Simple BM25 keyword-based retrieval."""
    
    def __init__(self, documents, k1=1.5, b=0.75):
        self.documents = documents
        self.k1 = k1
        self.b = b
        
        # Precompute
        self.doc_words = [doc.lower().split() for doc in documents]
        self.avg_dl = np.mean([len(d) for d in self.doc_words])
        self.n_docs = len(documents)
        
        # IDF scores
        self.idf = {}
        word_doc_count = defaultdict(int)
        for doc in self.doc_words:
            for word in set(doc):
                word_doc_count[word] += 1
        for word, count in word_doc_count.items():
            self.idf[word] = np.log((self.n_docs - count + 0.5) / (count + 0.5) + 1)
    
    def score(self, query, doc_idx):
        """Compute BM25 score for a query-document pair."""
        query_words = query.lower().split()
        doc = self.doc_words[doc_idx]
        dl = len(doc)
        
        score = 0
        word_freq = defaultdict(int)
        for w in doc:
            word_freq[w] += 1
        
        for word in query_words:
            if word in word_freq:
                tf = word_freq[word]
                idf = self.idf.get(word, 0)
                tf_norm = (tf * (self.k1 + 1)) / (tf + self.k1 * (1 - self.b + self.b * dl / self.avg_dl))
                score += idf * tf_norm
        return score
    
    def search(self, query, k=5):
        scores = [(i, self.score(query, i)) for i in range(self.n_docs)]
        scores.sort(key=lambda x: x[1], reverse=True)
        return [{'text': self.documents[i], 'score': s, 'index': i} for i, s in scores[:k]]


class HybridRetriever:
    """Combines dense (embedding) and sparse (BM25) retrieval."""
    
    def __init__(self, dense_retriever, bm25_retriever, alpha=0.5):
        self.dense = dense_retriever
        self.bm25 = bm25_retriever
        self.alpha = alpha  # weight for dense scores
    
    def retrieve(self, query, k=5):
        # Get candidates from both
        dense_results = self.dense.retrieve(query, k=k*2)
        bm25_results = self.bm25.search(query, k=k*2)
        
        # Normalize scores to [0, 1]
        def normalize(results):
            scores = [r['score'] for r in results]
            if max(scores) == min(scores):
                return {r['index'] if 'index' in r else i: 0.5 for i, r in enumerate(results)}
            return {r['index']: (r['score'] - min(scores)) / (max(scores) - min(scores)) 
                    for r in results}
        
        dense_scores = normalize(dense_results)
        bm25_scores = normalize(bm25_results)
        
        # Combine scores
        all_indices = set(dense_scores.keys()) | set(bm25_scores.keys())
        combined = []
        for idx in all_indices:
            d_score = dense_scores.get(idx, 0)
            b_score = bm25_scores.get(idx, 0)
            combined_score = self.alpha * d_score + (1 - self.alpha) * b_score
            combined.append({
                'text': chunks[idx],
                'score': combined_score,
                'metadata': chunk_metadata[idx],
                'index': idx,
                'dense_score': d_score,
                'bm25_score': b_score,
            })
        
        combined.sort(key=lambda x: x['score'], reverse=True)
        return combined[:k]


# Build hybrid retriever
bm25 = BM25Retriever(chunks)
hybrid = HybridRetriever(retriever, bm25, alpha=0.6)

# Compare retrieval results
test_q = "How does transfer learning work with convolutional networks?"
print('=== Dense Retrieval ===')
for r in retriever.retrieve(test_q, k=3):
    print(f'  [{r["score"]:.4f}] {r["metadata"]["topic"]}: {r["text"][:80]}...')

print('\n=== BM25 Retrieval ===')
for r in bm25.search(test_q, k=3):
    print(f'  [{r["score"]:.4f}] {r["text"][:80]}...')

print('\n=== Hybrid Retrieval ===')
for r in hybrid.retrieve(test_q, k=3):
    print(f'  [Combined: {r["score"]:.4f} | Dense: {r["dense_score"]:.4f} | BM25: {r["bm25_score"]:.4f}]')
    print(f'   {r["metadata"]["topic"]}: {r["text"][:80]}...')

### Comparing Retrieval Strategies

Let's run our evaluation suite on all three retrieval approaches.

In [ ]:
# Wrap BM25 and Hybrid to match Retriever interface
class BM25RetrieverWrapper:
    def __init__(self, bm25, metadata):
        self.bm25 = bm25
        self.metadata = metadata
    def retrieve(self, query, k=5):
        results = self.bm25.search(query, k=k)
        for r in results:
            r['metadata'] = self.metadata[r['index']]
        return results

bm25_wrapped = BM25RetrieverWrapper(bm25, chunk_metadata)

# Evaluate all three
strategies = {
    'TF-IDF Dense': retriever,
    'BM25 Sparse': bm25_wrapped,
    'Hybrid (0.6/0.4)': hybrid,
}

all_metrics = {}
for name, ret in strategies.items():
    all_metrics[name] = evaluate_retriever(ret, test_queries, k_values=[1, 3, 5])

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
k_vals = [1, 3, 5]
colors = ['#3498db', '#e74c3c', '#2ecc71']

for ax, metric_name in zip(axes, ['Hit Rate', 'MRR', 'Precision']):
    x = np.arange(len(k_vals))
    width = 0.25
    
    for i, (strat_name, m) in enumerate(all_metrics.items()):
        vals = [m[k][metric_name] for k in k_vals]
        bars = ax.bar(x + i*width, vals, width, label=strat_name, 
                      color=colors[i], edgecolor='white', linewidth=1.5)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.2f}', ha='center', fontsize=8, fontweight='bold')
    
    ax.set_title(metric_name, fontsize=14, fontweight='bold')
    ax.set_xticks(x + width)
    ax.set_xticklabels([f'k={k}' for k in k_vals])
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=9)

plt.suptitle('Retrieval Strategy Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

<a id='10'></a>
## 10. Key Takeaways

### What We Built
We implemented a **complete RAG pipeline from scratch** including:
- **3 chunking strategies**: Fixed-size, sentence-based, and overlapping windows
- **2 embedding approaches**: TF-IDF (sparse) and hash-based (dense)
- **A vector store** with cosine similarity search
- **3 retrieval strategies**: Dense, sparse (BM25), and hybrid
- **Re-ranking** for improved result quality
- **Comprehensive evaluation** with Hit Rate, MRR, and Precision metrics

### Key Lessons

1. **Chunk size matters enormously** -- too small loses context, too large adds noise. Overlapping chunks are often the best default.

2. **Hybrid search outperforms pure dense or sparse** -- combining semantic similarity with keyword matching covers both conceptual and lexical queries.

3. **Re-ranking improves precision** -- a two-stage retrieve-then-rerank pipeline is standard in production systems.

4. **Embedding quality is the bottleneck** -- upgrading from TF-IDF to sentence-transformers would dramatically improve results.

5. **Evaluation is essential** -- without metrics like MRR and Hit Rate, you can't systematically improve your RAG system.

### Production Recommendations

| Component | This Notebook | Production |
|-----------|--------------|------------|
| Embeddings | TF-IDF | sentence-transformers, OpenAI embeddings |
| Vector Store | NumPy array | FAISS, Pinecone, Weaviate |
| Chunking | Overlap window | Semantic chunking with LLM |
| Generation | Extractive | GPT-4, Claude, Llama |
| Re-ranking | Word overlap | Cross-encoder (ms-marco) |

---

## Further Reading

**Papers:**
- [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401) (Lewis et al., 2020) -- the original RAG paper
- [Lost in the Middle](https://arxiv.org/abs/2307.03172) (Liu et al., 2023) -- how LLMs use retrieved context
- [RAPTOR: Recursive Abstractive Processing for Tree-Organized Retrieval](https://arxiv.org/abs/2401.18059) -- hierarchical RAG

**Kaggle Notebooks & Competitions:**
- [LLM Science Exam](https://www.kaggle.com/competitions/kaggle-llm-science-exam) -- top solutions all use RAG
- [LECR](https://www.kaggle.com/competitions/learning-equality-curriculum-recommendations) -- content retrieval at scale

**Libraries to Explore:**
- [LangChain](https://python.langchain.com/) -- popular RAG framework
- [LlamaIndex](https://www.llamaindex.ai/) -- data framework for LLM apps
- [sentence-transformers](https://www.sbert.net/) -- state-of-the-art embeddings
- [FAISS](https://github.com/facebookresearch/faiss) -- production vector search

---

### **If you found this notebook helpful, please upvote! Your support helps the Kaggle community discover educational content.**

Feel free to fork this notebook and experiment with:
- Different chunk sizes and overlap ratios
- Adding `sentence-transformers` for better embeddings
- Connecting a real LLM for generation
- Testing on your own document corpus

**Connect with me** for more ML educational content!

In [ ]:
# Measure retrieval latency
n_trials = 50
latencies = {name: [] for name in strategies}

for _ in range(n_trials):
    q = test_queries[np.random.randint(len(test_queries))][0]
    for name, ret in strategies.items():
        start = time.time()
        ret.retrieve(q, k=5)
        latencies[name].append((time.time() - start) * 1000)  # ms

fig, ax = plt.subplots(figsize=(10, 5))
data = [latencies[name] for name in strategies]
bp = ax.boxplot(data, labels=list(strategies.keys()), patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Latency (ms)', fontsize=12)
ax.set_title('Retrieval Latency by Strategy', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

for name, lats in latencies.items():
    print(f'{name}: median={np.median(lats):.2f}ms, p95={np.percentile(lats, 95):.2f}ms')

### Re-Ranking

Re-ranking takes the initial retrieval results and re-scores them with a more accurate (but slower) model. Here we simulate a cross-encoder style re-ranker.

In [ ]:
class SimpleReRanker:
    """Re-ranks results using word overlap and position features."""
    
    def __init__(self):
        pass
    
    def rerank(self, query, results, top_k=3):
        query_words = set(query.lower().split())
        query_bigrams = set()
        qw = query.lower().split()
        for i in range(len(qw)-1):
            query_bigrams.add(f'{qw[i]} {qw[i+1]}')
        
        scored = []
        for r in results:
            text = r['text'].lower()
            text_words = set(text.split())
            
            # Features for re-ranking
            word_overlap = len(query_words & text_words) / max(len(query_words), 1)
            
            text_bigrams = set()
            tw = text.split()
            for i in range(len(tw)-1):
                text_bigrams.add(f'{tw[i]} {tw[i+1]}')
            bigram_overlap = len(query_bigrams & text_bigrams) / max(len(query_bigrams), 1)
            
            # Combined score (simulating a learned cross-encoder)
            rerank_score = 0.4 * r['score'] + 0.35 * word_overlap + 0.25 * bigram_overlap
            scored.append({**r, 'rerank_score': rerank_score, 'original_score': r['score']})
        
        scored.sort(key=lambda x: x['rerank_score'], reverse=True)
        return scored[:top_k]


reranker = SimpleReRanker()

# Demo re-ranking
query = "How do diffusion models generate images?"
initial = retriever.retrieve(query, k=10)
reranked = reranker.rerank(query, initial, top_k=5)

print(f'Query: "{query}"\n')
print('Before Re-Ranking (top 5):')
for i, r in enumerate(initial[:5]):
    print(f'  {i+1}. [{r["score"]:.4f}] {r["metadata"]["topic"]}')

print('\nAfter Re-Ranking (top 5):')
for i, r in enumerate(reranked):
    print(f'  {i+1}. [{r["rerank_score"]:.4f}] {r["metadata"]["topic"]} (was {r["original_score"]:.4f})')

<a id='10'></a>
## 10. 📝 Key Takeaways

### What We Built
We implemented a **complete RAG pipeline from scratch** including:
- **3 chunking strategies**: Fixed-size, sentence-based, and overlapping windows
- **2 embedding approaches**: TF-IDF (sparse) and hash-based (dense)
- **A vector store** with cosine similarity search
- **3 retrieval strategies**: Dense, sparse (BM25), and hybrid
- **Re-ranking** for improved result quality
- **Comprehensive evaluation** with Hit Rate, MRR, and Precision metrics

### Key Lessons

1. **Chunk size matters enormously** — too small loses context, too large adds noise. Overlapping chunks are often the best default.

2. **Hybrid search outperforms pure dense or sparse** — combining semantic similarity with keyword matching covers both conceptual and lexical queries.

3. **Re-ranking improves precision** — a two-stage retrieve-then-rerank pipeline is standard in production systems.

4. **Embedding quality is the bottleneck** — upgrading from TF-IDF to sentence-transformers would dramatically improve results.

5. **Evaluation is essential** — without metrics like MRR and Hit Rate, you can't systematically improve your RAG system.

### Production Recommendations

| Component | This Notebook | Production |
|-----------|--------------|------------|
| Embeddings | TF-IDF | sentence-transformers, OpenAI embeddings |
| Vector Store | NumPy array | FAISS, Pinecone, Weaviate |
| Chunking | Overlap window | Semantic chunking with LLM |
| Generation | Extractive | GPT-4, Claude, Llama |
| Re-ranking | Word overlap | Cross-encoder (ms-marco) |

---

**If you found this notebook helpful, please upvote!** Your support helps me create more educational content. 🙏

Feel free to fork this notebook and experiment with:
- Different chunk sizes and overlap ratios
- Adding sentence-transformers for better embeddings
- Connecting a real LLM for generation
- Testing on your own document corpus

## Portfolio Quality Addendum

### Objective
Build a retrieval-augmented generation pipeline that is factual, fast, and reproducible.

### Data
Document corpus chunks, embedding vectors, and query/answer evaluation sets.

### Method
Tune chunking, embedding retrieval, and reranking before generation prompt optimization.

### Evaluation
Measure retrieval hit-rate, answer faithfulness, and latency under fixed hardware budget.

### Insight and Trade-off
- Insight: Retrieval quality is the dominant bottleneck for final answer accuracy.
- Because generation cannot recover from missing evidence, weak recall propagates hallucinations.
- Therefore prioritize chunk strategy and reranker quality ahead of prompt complexity.
- Trade-off: larger context windows raise recall but can hurt latency and cost.
- Limitation: evaluation labels may not cover long-tail user intents.

## Conclusion and Next Steps

### Summary
RAG reliability improves most when retrieval engineering is treated as a first-class model component.

### Next Steps
1. Benchmark multiple embedding models with identical chunk policy.
2. Add citation-based faithfulness scoring to the eval loop.
3. Introduce caching and ANN index tuning for production latency.